# YOLOv8 Pothole Detection Training

This notebook trains a YOLOv8 object detection model using the single unified dataset we compiled from the Chitholian and Kaggle sources.

In [ ]:
!pip install ultralytics

In [ ]:
from ultralytics import YOLO
import os
from pathlib import Path

# Locate the dataset config generated in the preprocessing step
dataset_yaml_path = Path('../data/processed/pothole_yolo/dataset.yaml').resolve()
print(f"Dataset YAML: {dataset_yaml_path}")

In [ ]:
# Load a YOLOv8 pre-trained model (we use YOLOv8 small for a balance of speed and accuracy)
model = YOLO('yolov8s.pt')

In [ ]:
# Train the model. It automatically utilizes GPU if available natively.
# You may want to lower batch size to 8 if you run into memory errors during training.
results = model.train(
    data=str(dataset_yaml_path),
    epochs=50,
    imgsz=640,
    batch=16,
    name='pothole_yolov8s',
    project='../models', # Save results to the model directory automatically
    exist_ok=True
)

In [ ]:
# Validate using the hold-out validation set to check accuracy
metrics = model.val()
print("mAP50:", metrics.box.map50)

In [ ]:
# Example of running inference on a validation image
import glob
from IPython.display import display, Image

val_images = glob.glob(str(dataset_yaml_path.parent / "images" / "val" / "*.jpg"))
if not val_images:
    val_images = glob.glob(str(dataset_yaml_path.parent / "images" / "val" / "*.png"))

if val_images:
    sample_img = val_images[0]
    result = model.predict(sample_img, save=True, project='./runs')
    display(Image(filename=result[0].save_dir + '/' + Path(sample_img).name))